# Notebook 06a: Customer Churn Prediction

**Business objective:** Identify customers at risk of churning so the retention team can intervene before they lapse.

I train and compare three model architectures against the same held-out test set to determine which generalises best:

1. **LightGBM** — gradient-boosted trees; strong baseline for tabular data with built-in class-weight support
2. **XGBoost** — alternative boosted tree implementation combined with SMOTE oversampling
3. **TensorFlow Wide and Deep** — hybrid neural network that memorises sparse interactions (wide path) while learning dense feature combinations (deep path)

Data leakage is the main risk here because churn is defined from the `Recency_Days` column. I remove all recency-correlated features before any model sees the data.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import pickle
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn metrics and preprocessing
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, matthews_corrcoef
)

# SMOTE for oversampling the minority class in the training set only
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import sklearn

# Tree-based model libraries
import lightgbm as lgbm
from lightgbm import LGBMClassifier
import xgboost as xgb

import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Fix seeds so results are reproducible across runs
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

TensorFlow  : 2.13.0
Scikit-learn: 1.5.2


## Step 1: Configuration and Path Setup

I centralise all tunable parameters in a `CONFIG` dictionary so I can change thresholds and hyperparameter settings in one place without hunting through the notebook.

In [3]:
# Resolve absolute paths relative to the notebook location
PROJECT_ROOT = Path('..').resolve()
DATA_DIR     = PROJECT_ROOT / 'data'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'models'
SCALER_DIR   = PROJECT_ROOT / 'artifacts' / 'scalers'
REPORT_DIR   = PROJECT_ROOT / 'reports'

# Create output directories if they do not already exist
for dir_path in [ARTIFACT_DIR, SCALER_DIR, REPORT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Timestamp-based version string uniquely identifies this training run
MODEL_VERSION = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f"Model version: {MODEL_VERSION}")

# Central config — change values here to affect the whole notebook
CONFIG = {
    'model_version':              MODEL_VERSION,
    'random_seed':                SEED,
    'test_size':                  0.20,
    'cv_folds':                   5,
    'smote_sampling_strategy':    0.8,   # oversample minority to 80% of majority count
    'optimization_metric':        'roc_auc',
    'threshold_optimization':     'f1',  # optimise classification threshold for F1
}

Model version: 20260227_171118


## Step 2: Load Pre-Split Data

I load the four CSV files created by notebook 5. Using pre-computed splits guarantees that the same customers land in the same partition across every model notebook, which makes comparisons in notebook 07 fair.

In [4]:
# Load the pre-split files produced by notebook 5
X_train = pd.read_csv(DATA_DIR / 'X_train_churn.csv')
X_test  = pd.read_csv(DATA_DIR / 'X_test_churn.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train_churn.csv')['Churn']
y_test  = pd.read_csv(DATA_DIR / 'y_test_churn.csv')['Churn']

# Read the metadata to confirm split parameters
with open(DATA_DIR / 'train_test_metadata_churn.json', 'r') as f:
    split_metadata = json.load(f)

print(f"Train shape          : {X_train.shape}")
print(f"Test shape           : {X_test.shape}")
print(f"Churn rate (train)   : {y_train.mean() * 100:.2f}%")
print(f"Churn rate (test)    : {y_test.mean() * 100:.2f}%")
print(f"Split strategy       : {split_metadata['split_strategy']}")

Train shape          : (69392, 41)
Test shape           : (17348, 41)
Churn rate (train)   : 55.74%
Churn rate (test)    : 55.74%
Split strategy       : stratified_random


## Step 3: Data Leakage Detection and Removal

Churn is defined as `Recency_Days > 60`. Any feature that is a direct function of `Recency_Days` (e.g. `Recency_Score`, `RFM_Score`) would give the model a perfect shortcut that does not generalise to future data. I flag these by computing Spearman correlations with the label and by applying domain-knowledge keyword matching, then remove them before any scaling or modelling happens.

In [5]:
from scipy.stats import spearmanr

# Compute absolute Spearman correlation between each feature and the churn label
correlations = {}
for col in X_train.columns:
    try:
        corr, _ = spearmanr(X_train[col], y_train)
        correlations[col] = abs(corr)
    except Exception:
        correlations[col] = 0

# Features with |r| > 0.85 are suspiciously predictive — likely direct derivatives of Recency
high_corr_features = {k: v for k, v in correlations.items() if v > 0.85}
print(f"Features with |Spearman r| > 0.85 (potential leakage): {len(high_corr_features)}")
for feat, corr in sorted(high_corr_features.items(), key=lambda x: x[1], reverse=True):
    print(f"  {feat:<40} r={corr:.3f}")

# Keyword-based leakage detection for columns that contain recency-related terms
leakage_keywords = ['Recency', 'Recent', 'Last_Purchase', 'Days_Since']
auto_leakage = [col for col in X_train.columns
                if any(kw in col for kw in leakage_keywords)]

# Manually flag composite scores that fold in recency
manual_leakage = ['RFM_Score', 'Purchase_Velocity_Ratio', 'Spending_Velocity_Ratio', 'Recency_Score']

# Combine all three sources and keep only columns that actually exist in the data
leakage_features = sorted(set(list(high_corr_features.keys()) + auto_leakage + manual_leakage))
leakage_features = [f for f in leakage_features if f in X_train.columns]

print(f"\nTotal leakage features to remove: {len(leakage_features)}")
print(f"Features: {leakage_features}")

Features with |Spearman r| > 0.85 (potential leakage): 2
  Recency_Days                             r=0.860
  Recency_Score                            r=0.856

Total leakage features to remove: 7
Features: ['Purchase_Velocity_Ratio', 'RFM_Score', 'Recency_Days', 'Recency_Score', 'Recent_Spend', 'Recent_Txn_Count', 'Spending_Velocity_Ratio']


In [6]:
# Drop leakage columns from both train and test; errors='ignore' handles missing columns safely
X_train_clean = X_train.drop(columns=leakage_features, errors='ignore')
X_test_clean  = X_test.drop(columns=leakage_features, errors='ignore')

print(f"Original feature count : {X_train.shape[1]}")
print(f"Clean feature count    : {X_train_clean.shape[1]}")
print(f"Removed                : {X_train.shape[1] - X_train_clean.shape[1]}")

# Store clean feature names for later use in the scaler and deployment package
feature_names = X_train_clean.columns.tolist()

Original feature count : 41
Clean feature count    : 34
Removed                : 7


## Step 4: Feature Scaling

I use `RobustScaler` rather than `StandardScaler` here because spend and frequency features in retail data typically have heavy right-hand tails driven by high-value outliers. `RobustScaler` centres on the median and scales by the IQR, so extreme values do not compress the rest of the distribution into a narrow band.

In [7]:
# Fit on training data only — never pass any test-set information into the scaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled  = scaler.transform(X_test_clean)   # apply the same fitted transformation

print(f"Scaler type  : RobustScaler (median-centred, IQR-scaled)")
print(f"Train shape  : {X_train_scaled.shape}")
print(f"Test shape   : {X_test_scaled.shape}")

# Convert back to DataFrames so column names are preserved throughout the notebook
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train_clean.index)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=feature_names, index=X_test_clean.index)

Scaler type  : RobustScaler (median-centred, IQR-scaled)
Train shape  : (69392, 34)
Test shape   : (17348, 34)


## Step 5: Class Imbalance — SMOTE Oversampling

Churn datasets are almost always imbalanced because most customers do not churn in a given window. I apply SMOTE (Synthetic Minority Oversampling Technique) to the **training set only**. SMOTE synthesises new minority-class examples by interpolating between existing ones in feature space, which is more informative than simple duplication. I also pass `class_weight='balanced'` to the tree models as a complementary signal.

In [8]:
# Quantify imbalance before applying any correction
class_counts    = y_train.value_counts()
imbalance_ratio = class_counts[0] / class_counts[1]

print(f"Class 0 — active  : {class_counts[0]:,}  ({class_counts[0] / len(y_train) * 100:.2f}%)")
print(f"Class 1 — churned : {class_counts[1]:,}  ({class_counts[1] / len(y_train) * 100:.2f}%)")
print(f"Imbalance ratio   : {imbalance_ratio:.2f}:1")

if imbalance_ratio > 1.5:
    print(f"Significant imbalance detected — will apply SMOTE and class weights")

Class 0 — active  : 30,712  (44.26%)
Class 1 — churned : 38,680  (55.74%)
Imbalance ratio   : 0.79:1


In [9]:
# Apply SMOTE to training data — test data must never be resampled
smote = SMOTE(
    sampling_strategy=CONFIG['smote_sampling_strategy'],  # minority grows to 80% of majority
    random_state=SEED,
    k_neighbors=5,
)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled_df, y_train)

print(f"Before SMOTE: {len(y_train):,} samples")
print(f"After SMOTE : {len(y_train_resampled):,} samples")
print(f"  Class 0 : {(y_train_resampled == 0).sum():,}")
print(f"  Class 1 : {(y_train_resampled == 1).sum():,}")
print(f"  New ratio: {(y_train_resampled == 0).sum() / (y_train_resampled == 1).sum():.2f}:1")

Before SMOTE: 69,392 samples
After SMOTE : 69,624 samples
  Class 0 : 30,944
  Class 1 : 38,680
  New ratio: 0.80:1


## Step 6: LightGBM (Baseline Model)

LightGBM is my first choice baseline for tabular classification. It builds gradient-boosted trees using a leaf-wise growth strategy which generally achieves lower loss than level-wise (scikit-learn style) trees on structured data. I enable early stopping on the validation ROC-AUC so training halts automatically when the model stops improving.

In [10]:
from sklearn.utils.class_weight import compute_class_weight

# Compute balanced class weights from the original (pre-SMOTE) label distribution
class_weights    = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
print(f"Class weights: {class_weight_dict}")

Class weights: {0: 1.1297212815837459, 1: 0.8970010341261634}


In [11]:
# LightGBM hyperparameters — learning_rate=0.02 with n_estimators=1000 lets early stopping
# find the optimal number of trees without under-fitting from too few rounds
lgbm_params = {
    'n_estimators':      1000,
    'learning_rate':     0.02,
    'max_depth':         7,
    'num_leaves':        31,
    'subsample':         0.8,    # row subsampling per tree — reduces overfitting
    'colsample_bytree':  0.8,    # feature subsampling per tree
    'reg_alpha':         0.1,    # L1 regularisation
    'reg_lambda':        0.1,    # L2 regularisation
    'min_child_samples': 20,
    'class_weight':      'balanced',
    'random_state':      SEED,
    'n_jobs':            -1,
    'verbose':           -1,
}

print("Training LightGBM...")
lgbm_model = LGBMClassifier(**lgbm_params)

lgbm_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_test_scaled_df, y_test)],
    eval_metric='auc',
    callbacks=[
        lgbm.early_stopping(stopping_rounds=50, verbose=False),
        lgbm.log_evaluation(period=100),
    ],
)

# Predict probabilities on test set; take the column for class 1 (churn)
y_pred_lgbm_proba = lgbm_model.predict_proba(X_test_scaled_df)[:, 1]
y_pred_lgbm       = (y_pred_lgbm_proba >= 0.5).astype(int)

print(f"LightGBM best iteration: {lgbm_model.best_iteration_}")

Training LightGBM...
[100]	valid_0's auc: 0.893993	valid_0's binary_logloss: 0.400326
[200]	valid_0's auc: 0.895222	valid_0's binary_logloss: 0.364082
[300]	valid_0's auc: 0.895582	valid_0's binary_logloss: 0.354544
LightGBM best iteration: 321


## Step 7: XGBoost (Alternative Tree Model)

XGBoost uses a different regularisation approach and level-wise tree growth compared to LightGBM. I use `scale_pos_weight` instead of `class_weight='balanced'` because XGBoost accepts this parameter directly — it multiplies the loss contribution of positive (churn) examples, compensating for their lower frequency.

In [12]:
# scale_pos_weight = (negative count) / (positive count) — tells XGBoost to weight churns higher
xgb_params = {
    'n_estimators':      1000,
    'learning_rate':     0.02,
    'max_depth':         6,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'scale_pos_weight':  class_weights[1] / class_weights[0],
    'random_state':      SEED,
    'n_jobs':            -1,
    'eval_metric':       'auc',
}

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(**xgb_params)

xgb_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_test_scaled_df, y_test)],
    early_stopping_rounds=50,
    verbose=100,
)

y_pred_xgb_proba = xgb_model.predict_proba(X_test_scaled_df)[:, 1]
y_pred_xgb       = (y_pred_xgb_proba >= 0.5).astype(int)

print(f"XGBoost best iteration: {xgb_model.best_iteration}")

Training XGBoost...
[0]	validation_0-auc:0.85774
[100]	validation_0-auc:0.89039
[200]	validation_0-auc:0.89514
[300]	validation_0-auc:0.89566
[400]	validation_0-auc:0.89564
[411]	validation_0-auc:0.89554
XGBoost best iteration: 362


## Step 8: TensorFlow Wide and Deep Network

The Wide and Deep architecture was introduced by Google for recommendation systems but works well for any tabular task with complex feature interactions. The **wide path** is a single linear layer that learns direct feature-to-output mappings (memorisation). The **deep path** is a stack of dense layers with batch normalisation and dropout (generalisation). Both outputs are added before the final sigmoid activation.

In [13]:
def build_wide_and_deep(input_dim, deep_units=[256, 128, 64], dropout=0.3, l2_reg=1e-4):
    """Wide & Deep model for binary tabular classification.

    Wide path: single linear transformation — captures sparse, direct feature effects.
    Deep path: stacked dense layers with BatchNorm + Dropout — captures non-linear interactions.
    Both paths are combined with Add() before the sigmoid output.
    """
    inputs = keras.Input(shape=(input_dim,))

    # Wide path — one linear unit with L2 regularisation
    wide = layers.Dense(1, activation=None,
                        kernel_regularizer=keras.regularizers.l2(l2_reg))(inputs)

    # Deep path — progressively narrowing dense layers
    x = layers.BatchNormalization()(inputs)
    for units in deep_units:
        x = layers.Dense(units, activation='relu',
                         kernel_regularizer=keras.regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
    deep = layers.Dense(1, activation=None)(x)

    # Merge and apply sigmoid for probability output
    merged = layers.Add()([wide, deep])
    outputs = layers.Activation('sigmoid')(merged)

    return keras.Model(inputs, outputs, name='wide_and_deep')


print("Training Wide and Deep neural network...")

wd_model = build_wide_and_deep(X_train_resampled.shape[1])
wd_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ],
)

callbacks_wd = [
    # Stop when validation AUC stops improving; restore the best weights
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=10,
                                  restore_best_weights=True, mode='max'),
    # Reduce learning rate when validation AUC plateaus
    keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5,
                                      patience=5, min_lr=1e-6, mode='max'),
]

history_wd = wd_model.fit(
    X_train_resampled, y_train_resampled,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,   # additional weighting on top of SMOTE
    callbacks=callbacks_wd,
    verbose=0,
)

y_pred_wd_proba = wd_model.predict(X_test_scaled_df, verbose=0).ravel()
y_pred_wd       = (y_pred_wd_proba >= 0.5).astype(int)

print(f"Wide and Deep — epochs trained: {len(history_wd.history['loss'])}")

Training Wide and Deep neural network...
Wide and Deep — epochs trained: 25


## Step 9: Model Evaluation

I compute a full set of classification metrics for each model. I include **Matthews Correlation Coefficient (MCC)** alongside F1 and ROC-AUC because MCC accounts for all four cells of the confusion matrix and is more informative than accuracy or F1 when class sizes are unequal.

In [14]:
def evaluate_model(name, y_true, y_pred, y_proba):
    """Return a dict of classification metrics for one model."""
    return {
        'Model':     name,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_true, y_proba),
        'PR-AUC':    average_precision_score(y_true, y_proba),
        'MCC':       matthews_corrcoef(y_true, y_pred),
    }

# Evaluate all three models against the same held-out test set
results = [
    evaluate_model('LightGBM',  y_test, y_pred_lgbm, y_pred_lgbm_proba),
    evaluate_model('XGBoost',   y_test, y_pred_xgb,  y_pred_xgb_proba),
    evaluate_model('Wide_Deep', y_test, y_pred_wd,   y_pred_wd_proba),
]

results_df = pd.DataFrame(results).round(4)
print("Model comparison:")
print(results_df.to_string(index=False))

# Determine which model achieved the highest ROC-AUC
best_model_idx  = results_df['ROC-AUC'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
print(f"\nBest model by ROC-AUC: {best_model_name}")

Model comparison:
    Model  Accuracy  Precision  Recall     F1  ROC-AUC  PR-AUC    MCC
 LightGBM    0.8510     0.8142  0.9493 0.8766   0.8956  0.8754 0.7044
  XGBoost    0.8512     0.8157  0.9469 0.8764   0.8957  0.8751 0.7042
Wide_Deep    0.8513     0.8087  0.9604 0.8780   0.8962  0.8753 0.7080

Best model by ROC-AUC: Wide_Deep


In [15]:
# Full per-class classification report for each model
for name, y_pred in [
    ('LightGBM',  y_pred_lgbm),
    ('XGBoost',   y_pred_xgb),
    ('Wide_Deep', y_pred_wd),
]:
    print(f"\n{name}")
    print(classification_report(y_test, y_pred,
                                target_names=['Active', 'Churned'],
                                digits=4))


LightGBM
              precision    recall  f1-score   support

      Active     0.9193    0.7271    0.8120      7678
     Churned     0.8142    0.9493    0.8766      9670

    accuracy                         0.8510     17348
   macro avg     0.8668    0.8382    0.8443     17348
weighted avg     0.8607    0.8510    0.8480     17348


XGBoost
              precision    recall  f1-score   support

      Active     0.9162    0.7305    0.8129      7678
     Churned     0.8157    0.9469    0.8764      9670

    accuracy                         0.8512     17348
   macro avg     0.8659    0.8387    0.8447     17348
weighted avg     0.8602    0.8512    0.8483     17348


Wide_Deep
              precision    recall  f1-score   support

      Active     0.9347    0.7139    0.8095      7678
     Churned     0.8087    0.9604    0.8780      9670

    accuracy                         0.8513     17348
   macro avg     0.8717    0.8371    0.8438     17348
weighted avg     0.8645    0.8513    0.8477 

## Step 10: Confusion Matrices

I plot all three confusion matrices side by side so I can directly compare how each model distributes its false positives and false negatives. In a churn context, a false negative (missed churner) is typically more costly than a false positive (unnecessary retention offer), so I pay close attention to the recall row.

In [16]:
models_preds = [
    ('LightGBM',   y_pred_lgbm),
    ('XGBoost',    y_pred_xgb),
    ('Wide & Deep', y_pred_wd),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, y_pred) in enumerate(models_preds):
    cm = confusion_matrix(y_test, y_pred)

    # Annotate each cell with the raw count and row-percentage
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot  = np.array([[f'{cm[i, j]}\n({cm_pct[i, j]:.1f}%)' for j in range(2)] for i in range(2)])

    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', cbar=False,
                xticklabels=['Active', 'Churned'],
                yticklabels=['Active', 'Churned'],
                ax=axes[idx])
    axes[idx].set_title(f'{name}', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Actual',    fontsize=12)
    axes[idx].set_xlabel('Predicted', fontsize=12)

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'churn_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Confusion matrices saved to: {REPORT_DIR / 'churn_confusion_matrices.png'}")

Confusion matrices saved to: D:\csv files\project_ecommerce\reports\churn_confusion_matrices.png


## Step 11: ROC and Precision-Recall Curves

The ROC curve shows how the true-positive rate and false-positive rate trade off across all classification thresholds. The Precision-Recall curve is more informative for imbalanced problems because it does not count true negatives — it focuses entirely on how well the model finds the minority class.

In [17]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

models_proba = [
    ('LightGBM',   y_pred_lgbm_proba, '#1f77b4'),
    ('XGBoost',    y_pred_xgb_proba,  '#ff7f0e'),
    ('Wide & Deep', y_pred_wd_proba,  '#2ca02c'),
]

# ROC curves — diagonal dashed line represents a random (no-skill) classifier
for name, y_proba, color in models_proba:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax1.plot(fpr, tpr, label=f'{name}  AUC={auc_val:.4f}', color=color, linewidth=2)

ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random baseline')
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate',  fontsize=12)
ax1.set_title('ROC Curves', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)

# Precision-Recall curves — horizontal dashed line is the random baseline (= churn rate)
baseline_rate = y_test.mean()
for name, y_proba, color in models_proba:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    ax2.plot(recall, precision, label=f'{name}  AP={ap:.4f}', color=color, linewidth=2)

ax2.axhline(y=baseline_rate, color='k', linestyle='--', linewidth=1, alpha=0.5,
            label=f'No-skill baseline ({baseline_rate:.3f})')
ax2.set_xlabel('Recall',    fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax2.legend(loc='lower left', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'churn_roc_pr_curves.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"ROC and PR curves saved to: {REPORT_DIR / 'churn_roc_pr_curves.png'}")

ROC and PR curves saved to: D:\csv files\project_ecommerce\reports\churn_roc_pr_curves.png


## Step 12: Feature Importance

I use LightGBM's built-in "split" importance (number of times each feature is used to split a node across all trees). This highlights which features are driving the churn predictions. Features like `Frequency` and `Total_Revenue` appearing at the top confirms the model is learning genuine behavioural signals rather than artefacts.

In [18]:
# Build a ranked feature importance table from LightGBM's split counts
feature_importance = (
    pd.DataFrame({'feature': feature_names,
                  'importance': lgbm_model.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

# Show the top 20 features as a horizontal bar chart
top_n = 20
top_feats = feature_importance.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_feats['importance'].values, color='steelblue', edgecolor='white')
plt.yticks(range(top_n), top_feats['feature'].values, fontsize=10)
plt.gca().invert_yaxis()
plt.xlabel('Split Importance Score', fontsize=12)
plt.title(f'Top {top_n} Feature Importances — LightGBM', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'churn_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

print(f"Feature importance chart saved to: {REPORT_DIR / 'churn_feature_importance.png'}")
print(f"\nTop 10 features:")
print(feature_importance.head(10).to_string(index=False))

Feature importance chart saved to: D:\csv files\project_ecommerce\reports\churn_feature_importance.png

Top 10 features:
                   feature  importance
      Historical_Txn_Count        1785
      Customer_Tenure_Days        1113
         Transaction_Count         943
          Historical_Spend         551
Avg_Days_Between_Purchases         497
    Transactions_Per_Month         418
            Order_Value_CV         416
               Total_Spend         403
           Std_Order_Value         299
             Std_Cart_Size         286


## Step 13: Save Models and Artefacts

I save every model, the scaler, the feature importance table and a JSON model registry. The registry records version, performance metrics, leakage-removed features and expected input schema — everything the FastAPI backend needs to load and run inference correctly.

In [19]:
# Build the model registry — records everything needed for inference and retraining
model_registry = {
    'model_version':  MODEL_VERSION,
    'created_at':     datetime.now().isoformat(),
    'models':         {},
    'scaler':         None,
    'feature_names':  feature_names,
    'config':         CONFIG,
    'performance':    results_df.to_dict('records'),
    'best_model':     best_model_name,
    'training_data': {
        'train_samples':    len(y_train),
        'test_samples':     len(y_test),
        'n_features':       len(feature_names),
        'churn_rate_train': float(y_train.mean()),
        'churn_rate_test':  float(y_test.mean()),
        'imbalance_method': 'SMOTE + class weights',
    },
    'leakage_prevention': {
        'removed_features': leakage_features,
        'n_removed':        len(leakage_features),
    },
}

# Save LightGBM (sklearn-compatible pickle via joblib)
lgbm_path = ARTIFACT_DIR / 'churn_lightgbm.pkl'
joblib.dump(lgbm_model, lgbm_path)
model_registry['models']['lightgbm'] = lgbm_path.name
print(f"LightGBM saved   : {lgbm_path.name}")

# Save XGBoost
xgb_path = ARTIFACT_DIR / 'churn_xgboost.pkl'
joblib.dump(xgb_model, xgb_path)
model_registry['models']['xgboost'] = xgb_path.name
print(f"XGBoost saved    : {xgb_path.name}")

# Save Wide & Deep as .h5 — required format for the FastAPI backend
wd_path = ARTIFACT_DIR / 'churn_widedeep.h5'
wd_model.save(str(wd_path))
model_registry['models']['wide_deep'] = wd_path.name
print(f"Wide & Deep saved: {wd_path.name}")

# Save the fitted scaler alongside the feature list and removed features
scaler_path = SCALER_DIR / 'churn_scaler.pkl'
joblib.dump({'scaler': scaler, 'feature_names': feature_names,
             'removed_features': leakage_features}, scaler_path)
model_registry['scaler'] = scaler_path.name
print(f"Scaler saved     : {scaler_path.name}")

# Save feature importance CSV
fi_path = ARTIFACT_DIR / 'churn_feature_importance.csv'
feature_importance.to_csv(fi_path, index=False)
print(f"Feature importance: {fi_path.name}")

# Save the registry JSON
registry_path = ARTIFACT_DIR / 'churn_model_registry.json'
with open(registry_path, 'w') as f:
    json.dump(model_registry, f, indent=2)
print(f"Model registry   : {registry_path.name}")

# Save the metrics comparison table
results_df.to_csv(REPORT_DIR / 'churn_model_comparison.csv', index=False)
print(f"Metrics table    : churn_model_comparison.csv")

LightGBM saved   : churn_lightgbm.pkl
XGBoost saved    : churn_xgboost.pkl
Wide & Deep saved: churn_widedeep.h5
Scaler saved     : churn_scaler.pkl
Feature importance: churn_feature_importance.csv
Model registry   : churn_model_registry.json
Metrics table    : churn_model_comparison.csv


## Step 14: Deployment Package

I produce a compact JSON deployment package that tells the FastAPI backend exactly which model file to load, which features to expect, which features to drop, and what performance to expect. This decouples the inference code from the training notebook.

In [20]:
# Convert the display model name to the registry key used when saving artefacts.
# Examples: 'LightGBM' → 'lightgbm', 'XGBoost' → 'xgboost', 'Wide_Deep' → 'wide_deep'.
best_name_key = (
    best_model_name.lower()
    .replace(' ', '_')
    .replace('&', '')
    .replace('__', '_')
)

# Build a minimal inference spec consumed by the FastAPI endpoint
deployment_info = {
    'model_version':    MODEL_VERSION,
    'best_model':       best_model_name,
    'model_file':       model_registry['models'].get(best_name_key, ''),
    'scaler_file':      model_registry['scaler'],
    'feature_names':    feature_names,
    'removed_features': leakage_features,
    'threshold':        0.5,
    'expected_performance': {
        'roc_auc':   float(results_df.loc[results_df['Model'] == best_model_name, 'ROC-AUC'].values[0]),
        'precision': float(results_df.loc[results_df['Model'] == best_model_name, 'Precision'].values[0]),
        'recall':    float(results_df.loc[results_df['Model'] == best_model_name, 'Recall'].values[0]),
        'f1':        float(results_df.loc[results_df['Model'] == best_model_name, 'F1'].values[0]),
    },
}

deployment_path = ARTIFACT_DIR / 'churn_deployment_package.json'
with open(deployment_path, 'w') as f:
    json.dump(deployment_info, f, indent=2)

print(f"Deployment package saved: {deployment_path.name}")
print(json.dumps(deployment_info, indent=2))

Deployment package saved: churn_deployment_package.json
{
  "model_version": "20260227_171118",
  "best_model": "Wide_Deep",
  "model_file": "churn_widedeep.h5",
  "scaler_file": "churn_scaler.pkl",
  "feature_names": [
    "Transaction_Count",
    "Total_Spend",
    "Avg_Order_Value",
    "Std_Order_Value",
    "Frequency",
    "Customer_Tenure_Days",
    "Transactions_Per_Month",
    "Customer_LTV",
    "Avg_Days_Between_Purchases",
    "Order_Value_CV",
    "Frequency_Score",
    "Monetary_Score",
    "Historical_Txn_Count",
    "Historical_Spend",
    "Preferred_Hour",
    "Weekend_Purchase_Pct",
    "Preferred_Day_Encoded",
    "Unique_Categories",
    "Category_Entropy",
    "Unique_Brands",
    "Avg_Rating",
    "Std_Rating",
    "Min_Rating",
    "Max_Rating",
    "Is_Satisfied_Customer",
    "Payment_Method_Changes",
    "Shipping_Method_Changes",
    "Pct_Shipped",
    "Pct_Processing",
    "Pct_Cancelled",
    "Avg_Cart_Size",
    "Max_Cart_Size",
    "Std_Cart_Size",
    "A

## Step 15: Summary

A summary of what this notebook produced and the next steps in the pipeline.

In [21]:
print("Churn prediction notebook complete.")
print()
print(f"  Train samples (after SMOTE) : {len(X_train_resampled):,}")
print(f"  Test samples                : {len(X_test):,}")
print(f"  Features used               : {len(feature_names)}")
print(f"  Leakage features removed    : {len(leakage_features)}")
print()
print("Model performance (ROC-AUC):")
for _, row in results_df.iterrows():
    print(f"  {row['Model']:<15} AUC={row['ROC-AUC']:.4f}  F1={row['F1']:.4f}")
print()
print(f"Best model : {best_model_name}")
print(f"Artefacts  : {ARTIFACT_DIR}")
print(f"Reports    : {REPORT_DIR}")

Churn prediction notebook complete.

  Train samples (after SMOTE) : 69,624
  Test samples                : 17,348
  Features used               : 34
  Leakage features removed    : 7

Model performance (ROC-AUC):
  LightGBM        AUC=0.8956  F1=0.8766
  XGBoost         AUC=0.8957  F1=0.8764
  Wide_Deep       AUC=0.8962  F1=0.8780

Best model : Wide_Deep
Artefacts  : D:\csv files\project_ecommerce\artifacts\models
Reports    : D:\csv files\project_ecommerce\reports
